# Aboveground Carbon Across Biomes

Compares aboveground carbon density (Mg C/ha) across four contrasting forest types using two independent satellite-derived sources (ESA CCI Biomass and GEDI L4B), plus a complementary structural comparison using Google's AlphaEarth satellite embeddings.

**Zones:** boreal managed forest (Abitibi, Quebec), intact tropical rainforest (Tapajos, Brazil), native temperate forest (Alerce Costero, Chile), and an even-aged Pinus radiata plantation (Biobio, Chile).

**Pipeline:** define zones -> carbon from ESA CCI Biomass -> carbon from GEDI L4B -> cross-source comparison chart -> AlphaEarth zone signatures -> similarity heatmap.

In [ ]:
import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')

from zones import ZONES, get_zone_geometry, get_zone_label
from carbon_sources import get_esa_cci_carbon, get_gedi_carbon, mean_carbon_over_zone
from embedding_utils import get_mean_embedding, cosine_similarity_matrix

PROJECT = "your-gee-project-id"
ee.Initialize(project=PROJECT)

## 1. Study zones

Small bounding boxes (~20-25 km) — illustrative placeholders centered on well-known sites for each forest type. Adjust in `src/zones.py` if you have more precise boundaries.

In [ ]:
for key in ZONES:
    geom = get_zone_geometry(key)
    area_ha = geom.area().divide(10000).getInfo()
    print(f"{key}: {get_zone_label(key)} — {area_ha:,.0f} ha")

## 2. Aboveground carbon — ESA CCI Biomass (2022)

Continuous, gap-free 100 m maps. Citation: Santoro & Cartus (2025), ESA CCI Biomass v6.0.

In [ ]:
YEAR = 2022  # most recent year available in ESA CCI Biomass v6.0

esa_results = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    carbon_img = get_esa_cci_carbon(YEAR, geom)
    mean_carbon = mean_carbon_over_zone(carbon_img, geom, scale=100)
    esa_results[key] = mean_carbon
    print(f"{key}: {mean_carbon:,.1f} Mg C/ha (ESA CCI Biomass)")

## 3. Aboveground carbon — GEDI L4B

Spaceborne lidar, aggregated across the mission period (not a single calendar year) — a cross-check against ESA CCI, not a like-for-like year match.

In [ ]:
gedi_results = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    carbon_img = get_gedi_carbon(geom)
    mean_carbon = mean_carbon_over_zone(carbon_img, geom, scale=1000)
    gedi_results[key] = mean_carbon
    print(f"{key}: {mean_carbon:,.1f} Mg C/ha (GEDI L4B)")

## 4. Result 1: cross-source carbon comparison

The headline chart for the LinkedIn post — two independent sources, same four zones.

In [ ]:
comparison_df = pd.DataFrame({
    "zone": [get_zone_label(k) for k in ZONES],
    "ESA CCI Biomass": [esa_results[k] for k in ZONES],
    "GEDI L4B": [gedi_results[k] for k in ZONES],
})
comparison_df.to_csv("../figures/carbon_comparison.csv", index=False)
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(comparison_df))
width = 0.35

ax.bar(x - width/2, comparison_df["ESA CCI Biomass"], width, label="ESA CCI Biomass", color="forestgreen")
ax.bar(x + width/2, comparison_df["GEDI L4B"], width, label="GEDI L4B", color="saddlebrown")

ax.set_ylabel("Aboveground carbon (Mg C/ha)")
ax.set_title("Aboveground carbon across biomes — two independent sources")
ax.set_xticks(x)
ax.set_xticklabels(comparison_df["zone"], rotation=20, ha="right")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/carbon_comparison.png", dpi=200)
plt.show()

## 5. AlphaEarth zone signatures

A complementary structural check: how distinct are these four sites in AlphaEarth's 64-dimensional embedding space? An even-aged plantation is expected to look more internally uniform and more different from a structurally complex native forest than two native forests would look from each other.

In [ ]:
EMBEDDING_YEAR = 2023

embeddings = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    embeddings[key] = get_mean_embedding(EMBEDDING_YEAR, geom)
    print(f"{key}: embedding vector computed ({len(embeddings[key])} dims)")

## 6. Result 2: zone similarity heatmap for LinkedIn

In [ ]:
labels, sim_matrix = cosine_similarity_matrix(embeddings)
display_labels = [get_zone_label(k) for k in labels]

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(sim_matrix, cmap="YlGnBu", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(display_labels, rotation=30, ha="right")
ax.set_yticklabels(display_labels)
ax.set_title("AlphaEarth embedding similarity between zones")

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                 color="white" if sim_matrix[i, j] > 0.5 else "black")

plt.colorbar(im, label="Cosine similarity")
plt.tight_layout()
plt.savefig("../figures/zone_similarity_heatmap.png", dpi=200)
plt.show()

## 7. Combined result for LinkedIn

Headline numbers for the post caption.

In [ ]:
for key in ZONES:
    print(f"{get_zone_label(key)}:")
    print(f"  ESA CCI Biomass: {esa_results[key]:,.1f} Mg C/ha")
    print(f"  GEDI L4B:        {gedi_results[key]:,.1f} Mg C/ha")